# 📊 Notebook 01 — Exploratory Data Analysis & Planning Accuracy
**FMCG Promotional Analytics | Portfolio Project**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Panosemmanouilidis2/fmcg-ds-technical-portfolio/blob/main/promotional-analytics/notebooks/01_eda_planning_accuracy.ipynb)

This notebook explores a synthesised promotional dataset covering two markets.  
It analyses the distribution of sell-out volumes, planning accuracy, and key drivers of promotional performance.

| | |
|---|---|
| Data | `sample_synthetic.csv` — 5,000 rows, 39 columns |
| Markets | Market A, Market B |
| Target | `ActualSellOutVolume` — actual units sold during a promotion |
| Next | `02_feature_engineering.ipynb` |


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 1 — IMPORTS
# ══════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')

os.makedirs('results', exist_ok=True)
os.makedirs('data',    exist_ok=True)
print("✅ Libraries loaded")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 2 — LOAD DATA (from GitHub — no upload needed)
# ══════════════════════════════════════════════════════════════════
CSV_URL = "https://raw.githubusercontent.com/Panosemmanouilidis2/fmcg-ds-technical-portfolio/main/promotional-analytics/notebooks/sample_synthetic.csv"

df = pd.read_csv(CSV_URL)

# Rename markets to generic labels
df['Market'] = df['Market'].map({
    'Western Europe' : 'Market A',
    'Southeast Asia' : 'Market B'
})

print(f"Rows    : {len(df):,}")
print(f"Columns : {df.shape[1]}")
print(f"\nMarket split:")
print(df['Market'].value_counts())


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 3 — BASIC DATA PROFILE
# ══════════════════════════════════════════════════════════════════
print("=== COLUMN TYPES ===")
print(df.dtypes.value_counts())

print("\n=== CATEGORICAL COLUMNS — UNIQUE VALUES ===")
cat_cols = df.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    vals = df[col].unique()
    print(f"  {col:<25} {df[col].nunique():>4} unique  {list(vals[:5])}")

print(f"\n=== MISSING VALUES ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values ✅")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 4 — TARGET VARIABLE ANALYSIS
# ══════════════════════════════════════════════════════════════════
TARGET = 'ActualSellOutVolume'

print("=== TARGET VARIABLE: ActualSellOutVolume ===")
print(df[TARGET].describe().round(1))
print(f"\nSkewness  : {df[TARGET].skew():.2f}  (>1 = right-skewed, needs log transform)")
print(f"Zero rows : {(df[TARGET] == 0).sum()}")
print(f"Neg rows  : {(df[TARGET] < 0).sum()}")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 5 — TARGET DISTRIBUTION: RAW vs LOG
# A right-skewed target makes linear models perform poorly.
# Log transforming compresses the long tail and normalises the shape,
# which helps all ML algorithms learn more effectively.
# ══════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[TARGET].clip(upper=200_000), bins=60,
             color='#378ADD', edgecolor='white', linewidth=0.4)
axes[0].set_title(f'Raw Sell-Out Volume  (skew={df[TARGET].skew():.1f})', fontsize=12)
axes[0].set_xlabel('Units (clipped at 200k)')
axes[0].set_ylabel('Count')

log_target = np.log1p(df[TARGET])
axes[1].hist(log_target, bins=60,
             color='#1D9E75', edgecolor='white', linewidth=0.4)
axes[1].set_title(f'Log-Transformed  (skew={log_target.skew():.2f})', fontsize=12)
axes[1].set_xlabel('log(Units + 1)')
axes[1].set_ylabel('Count')

plt.suptitle('Sell-Out Volume Distribution — Before and After Log Transform',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('results/01_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved → results/01_target_distribution.png")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 6 — SELL-OUT BY MARKET
# ══════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, market, color in zip(axes,
                              ['Market A', 'Market B'],
                              ['#378ADD', '#E87B2E']):
    subset = df[df['Market'] == market][TARGET]
    ax.hist(np.log1p(subset), bins=50, color=color,
            edgecolor='white', linewidth=0.4, alpha=0.9)
    ax.set_title(f'{market} — Log Sell-Out  (n={len(subset):,})', fontsize=12)
    ax.set_xlabel('log(Units + 1)')
    ax.set_ylabel('Count')
    ax.axvline(np.log1p(subset.median()), color='black',
               linestyle='--', linewidth=1.2,
               label=f'Median = {subset.median():,.0f}')
    ax.legend(fontsize=9)

plt.suptitle('Sell-Out Distribution by Market', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('results/01_sellout_by_market.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 7 — PLANNING ACCURACY ANALYSIS
# PlanningAccuracyRatio = ActualSellOut / PlannedSellIn
# Ratio > 1 : under-forecast (more sold than planned)
# Ratio < 1 : over-forecast  (less sold than planned)
# ══════════════════════════════════════════════════════════════════
ratio = df['PlanningAccuracyRatio']

print("=== PLANNING ACCURACY RATIO ===")
print(ratio.describe(percentiles=[.05, .25, .5, .75, .95]).round(3))
print(f"\nRatio > 2  (severe under-forecast) : {(ratio > 2).sum():,} rows ({(ratio > 2).mean()*100:.1f}%)")
print(f"Ratio < 0.5 (severe over-forecast) : {(ratio < 0.5).sum():,} rows ({(ratio < 0.5).mean()*100:.1f}%)")
print(f"Ratio 0.8-1.2 (accurate)           : {((ratio>=0.8)&(ratio<=1.2)).sum():,} rows ({((ratio>=0.8)&(ratio<=1.2)).mean()*100:.1f}%)")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 8 — PLANNING ACCURACY BY MARKET
# ══════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, market, color in zip(axes,
                              ['Market A', 'Market B'],
                              ['#378ADD', '#E87B2E']):
    subset = df[df['Market'] == market]['PlanningAccuracyRatio'].clip(0, 5)
    ax.hist(subset, bins=50, color=color, edgecolor='white',
            linewidth=0.4, alpha=0.9)
    ax.axvline(1.0, color='red', linestyle='--', linewidth=1.5,
               label='Perfect (ratio=1)')
    ax.axvline(subset.median(), color='black', linestyle='-',
               linewidth=1.2, label=f'Median = {subset.median():.2f}')
    ax.set_title(f'{market} — Planning Accuracy Ratio', fontsize=12)
    ax.set_xlabel('Actual / Planned (clipped at 5)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

plt.suptitle('Planning Accuracy Ratio by Market  (1.0 = perfect forecast)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('results/01_planning_accuracy_by_market.png', dpi=150,
            bbox_inches='tight')
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 9 — MEDIAN PLANNING ACCURACY BY PROMO MECHANIC
# ══════════════════════════════════════════════════════════════════
mech_acc = (df.groupby('PromoMechanic')['PlanningAccuracyRatio']
              .median()
              .sort_values()
              .clip(0, 4))

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#D85A30' if v < 0.8 else '#1D9E75' if v > 1.2 else '#378ADD'
          for v in mech_acc]
mech_acc.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.axvline(1.0, color='black', linestyle='--', linewidth=1.2,
           label='Perfect = 1.0')
ax.set_title('Median Planning Accuracy Ratio by Promo Mechanic', fontsize=13)
ax.set_xlabel('Median Actual / Planned')
ax.legend()
plt.tight_layout()
plt.savefig('results/01_accuracy_by_mechanic.png', dpi=150, bbox_inches='tight')
plt.show()

print("Green = under-forecast (actual > planned)")
print("Red   = over-forecast  (actual < planned)")
print("Blue  = broadly accurate (0.8 – 1.2)")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 10 — CORRELATION: PLANNED FEATURES vs ACTUAL SELL-OUT
# Only planned (pre-execution) columns used — no data leakage
# ══════════════════════════════════════════════════════════════════
planned_cols = [
    'PlannedPromoSalesVolumeSellIn', 'PlannedBaselineVolume',
    'PlannedIncrementalVolume',      'PlannedUpliftRate',
    'PlannedNetPromoGSVSellIn',      'PlannedTTSTotal',
    'PlannedCostPerUnit',            'PlannedROI_proxy',
    'PromoDurationWeeks',            'ListPrice',
    TARGET
]

corr = df[planned_cols].corr()[[TARGET]].drop(TARGET).sort_values(TARGET)

fig, ax = plt.subplots(figsize=(7, 6))
colors = ['#D85A30' if v < 0 else '#1D9E75' for v in corr[TARGET]]
corr[TARGET].plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_title('Correlation with Actual Sell-Out Volume\n(planned features only)',
             fontsize=12)
ax.set_xlabel('Pearson Correlation')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('results/01_correlation_with_target.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 11 — SAVE CLEANED DATASET FOR NOTEBOOK 02
# ══════════════════════════════════════════════════════════════════
df.to_csv('data/df_clean.csv', index=False)

print("✅ Saved → data/df_clean.csv")
print(f"   {len(df):,} rows × {df.shape[1]} columns")
print()
print("=== EDA SUMMARY ===")
print(f"Target skewness    : {df[TARGET].skew():.1f}  → log transform required")
print(f"No missing values  : ✅")
for market in ['Market A','Market B']:
    med = df[df['Market']==market]['PlanningAccuracyRatio'].median()
    direction = 'under-forecast' if med > 1 else 'over-forecast'
    print(f"{market} median accuracy : {med:.2f}  ({direction})")
print()
print("▶ Next: run 02_feature_engineering.ipynb")


## 📝 EDA Findings Summary

| Finding | Detail |
|---|---|
| Target skew | Heavily right-skewed — log transform essential before modelling |
| Planning accuracy | Both markets miss plan; directions differ by market |
| Strongest predictor | `PlannedBaselineVolume` — high baseline promos sell more |
| Weakest mechanics | Pipe Fill and Other show the largest planning errors |
| No missing values | Dataset is clean — no imputation required |

**Next step:** `02_feature_engineering.ipynb`
